# Forecast Visualization and Economic Context Dashboard

Interactive notebook for communicating forecast quality and macro-financial narratives.

Features:
- Actual vs forecast overlays for key models
- Cumulative absolute-error comparison
- Volatility-period zooming
- Economic event annotation template

In [14]:
import os
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if os.path.basename(os.getcwd()) == 'khanh_model_analysis':
    os.chdir('..')

sys.path.append('src')
from statistical_validation import load_config, discover_forecasts, event_annotation_frame

config = load_config('configs/pipeline_config.yaml')
active_target = config['active_target']
forecasts = discover_forecasts(config)

if forecasts.empty:
    raise ValueError('No forecast files found under results/<active_target>/<model>/forecasts.csv')

forecasts['Date'] = pd.to_datetime(forecasts['Date'])
if 'AE' not in forecasts.columns:
    if 'Error' in forecasts.columns:
        forecasts['AE'] = forecasts['Error'].abs()
    else:
        forecasts['AE'] = (forecasts['Actual'] - forecasts['Forecast']).abs()

preferred_models = ['hybrid_arima_mlp', 'hybrid_arima_svr', 'var']
available_models = sorted(forecasts['Model'].dropna().unique().tolist())
top_models = [m for m in preferred_models if m in available_models]

if len(top_models) < 3:
    ranked = (
        forecasts[forecasts['Set'] == 'test']
        .groupby('Model', as_index=False)['AE']
        .mean()
        .sort_values('AE')
    )
    for m in ranked['Model']:
        if m not in top_models:
            top_models.append(m)
    top_models = top_models[:3]

if not top_models:
    raise ValueError('No models available for plotting after discovery/filtering.')

plot_df = forecasts[(forecasts['Set'] == 'test') & (forecasts['Model'].isin(top_models))].copy()
if plot_df.empty:
    raise ValueError('No test-set forecast rows available for selected models.')

print('Active target:', active_target)
print('Top models:', top_models)
print('Plot rows:', len(plot_df))

Active target: PHP
Top models: ['hybrid_arima_mlp', 'hybrid_arima_svr', 'var']
Plot rows: 6345


#### Interpretation
This cell loads data for visualization and confirms active modeling scope. Verify sample coverage before interpreting any chart-level conclusion.

In [10]:
for pair in sorted(plot_df['Pair'].unique()):
    sub = plot_df[plot_df['Pair'] == pair].sort_values('Date')
    fig = go.Figure()
    actual = sub[sub['Model'] == top_models[0]][['Date', 'Actual']].drop_duplicates()
    fig.add_trace(go.Scatter(x=actual['Date'], y=actual['Actual'], mode='lines', name='Actual', line=dict(color='black', width=2)))
    for m in top_models:
        msub = sub[sub['Model'] == m]
        fig.add_trace(go.Scatter(x=msub['Date'], y=msub['Forecast'], mode='lines', name=m))
    fig.update_layout(template='plotly_white', title=f'{pair}: forecast dashboard', width=1150, height=450)
    fig.show()

#### Interpretation
The first plot compares forecast levels against realized series. Better models track turning points with smaller lag and reduced overshooting.

In [11]:
err = plot_df.copy()
err['abs_error'] = err['Error'].abs()
err['cum_abs_error'] = err.sort_values('Date').groupby(['Pair', 'Model'])['abs_error'].cumsum()

events = event_annotation_frame()
events['Date'] = pd.to_datetime(events['Date'])

for pair in sorted(err['Pair'].unique()):
    sub = err[err['Pair'] == pair]
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for m in top_models:
        msub = sub[sub['Model'] == m].sort_values('Date')
        fig.add_trace(go.Scatter(x=msub['Date'], y=msub['Forecast'], mode='lines', name=f'{m} forecast'), secondary_y=False)
    actual = sub[sub['Model'] == top_models[0]][['Date', 'Actual']].drop_duplicates().sort_values('Date')
    fig.add_trace(go.Scatter(x=actual['Date'], y=actual['Actual'], mode='lines', name='Actual', line=dict(color='black', width=2)), secondary_y=False)
    vol_proxy = sub.groupby('Date', as_index=False)['abs_error'].mean().sort_values('Date')
    fig.add_trace(go.Scatter(x=vol_proxy['Date'], y=vol_proxy['abs_error'], mode='lines', name='Realized abs error', line=dict(dash='dot', color='firebrick')), secondary_y=True)
    for _, ev in events.iterrows():
        fig.add_vline(x=ev['Date'], line_dash='dot', line_color='gray', opacity=0.35)
        fig.add_annotation(x=ev['Date'], y=1.02, yref='paper', text=ev['Event'], showarrow=False, font=dict(size=9), textangle=-90)
    fig.update_layout(
        template='plotly_white',
        title=f'{pair}: actual vs forecast with event markers',
        width=1300,
        height=550,
        xaxis=dict(rangeslider=dict(visible=True)),
        legend=dict(orientation='h')
    )
    fig.update_yaxes(title_text='Log-return (%)', secondary_y=False)
    fig.update_yaxes(title_text='Realized abs error', secondary_y=True)
    fig.show()

#### Interpretation
This panel visualizes forecast errors through time and by model. Systematic bias or clustered large errors can indicate regime-dependent misspecification.

## Uncertainty Snapshot: Simple Fan Chart (Test Window)

This example builds an empirical uncertainty band from rolling forecast errors and overlays 50% and 90% intervals around a selected model forecast path.

In [15]:
fan_model = top_models[0]
fan_pair = sorted(plot_df['Pair'].unique())[0]

fan_sub = plot_df[(plot_df['Model'] == fan_model) & (plot_df['Pair'] == fan_pair)].sort_values('Date').copy()
fan_sub['err'] = fan_sub['Actual'] - fan_sub['Forecast']

# Rolling empirical dispersion from recent forecast errors.
roll_std = fan_sub['err'].rolling(window=20, min_periods=10).std().bfill()
z50 = 0.67448975
z90 = 1.64485363

fan_sub['lo50'] = fan_sub['Forecast'] - z50 * roll_std
fan_sub['hi50'] = fan_sub['Forecast'] + z50 * roll_std
fan_sub['lo90'] = fan_sub['Forecast'] - z90 * roll_std
fan_sub['hi90'] = fan_sub['Forecast'] + z90 * roll_std

fig_fan = go.Figure()
fig_fan.add_trace(go.Scatter(
    x=fan_sub['Date'], y=fan_sub['hi90'], mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'
))
fig_fan.add_trace(go.Scatter(
    x=fan_sub['Date'], y=fan_sub['lo90'], mode='lines', line=dict(width=0),
    fill='tonexty', fillcolor='rgba(70,130,180,0.18)', name='90% band', hoverinfo='skip'
))
fig_fan.add_trace(go.Scatter(
    x=fan_sub['Date'], y=fan_sub['hi50'], mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'
))
fig_fan.add_trace(go.Scatter(
    x=fan_sub['Date'], y=fan_sub['lo50'], mode='lines', line=dict(width=0),
    fill='tonexty', fillcolor='rgba(70,130,180,0.30)', name='50% band', hoverinfo='skip'
))
fig_fan.add_trace(go.Scatter(
    x=fan_sub['Date'], y=fan_sub['Forecast'], mode='lines', name=f'{fan_model} forecast', line=dict(color='steelblue', width=2)
))
fig_fan.add_trace(go.Scatter(
    x=fan_sub['Date'], y=fan_sub['Actual'], mode='lines', name='Actual', line=dict(color='black', width=2)
))

fig_fan.update_layout(
    template='plotly_white',
    title=f'{fan_pair}: uncertainty fan chart for {fan_model} (test)',
    width=1200, height=500
 )
fig_fan.show()

fan_out = fan_sub[['Date', 'Pair', 'Model', 'Actual', 'Forecast', 'lo50', 'hi50', 'lo90', 'hi90']].copy()
fan_out.to_csv(f"results/{active_target}/evaluation/fan_chart_{fan_pair}_{fan_model}.csv", index=False)
print('Saved:', f"results/{active_target}/evaluation/fan_chart_{fan_pair}_{fan_model}.csv")

Saved: results/PHP/evaluation/fan_chart_CNYPHP_RET_hybrid_arima_mlp.csv


## Economic Event Overlay Template

Add a CSV with columns `Date`, `Event`, `Category` (policy, risk, trade) and merge with forecast-error spikes.

Suggested events to annotate for PHP analysis:
- BSP policy-rate surprises
- Federal Reserve policy shifts
- High-VIX global risk-off episodes
- Major China growth/trade surprises
- Balance-of-payments and remittance shocks

In [16]:
out_dir = f'results/{active_target}/evaluation'
os.makedirs(out_dir, exist_ok=True)

if 'err' not in globals():
    err = plot_df.copy()
    err['abs_error'] = err['Error'].abs()
    err['cum_abs_error'] = err.sort_values('Date').groupby(['Pair', 'Model'])['abs_error'].cumsum()

err.to_csv(f'{out_dir}/forecast_dashboard_error_paths.csv', index=False)
print('Saved:', f'{out_dir}/forecast_dashboard_error_paths.csv')

Saved: results/PHP/evaluation/forecast_dashboard_error_paths.csv


#### Interpretation
The saved figure confirmation indicates all visualization artifacts were exported. These files should be used as canonical plots in the final narrative.

### Narrative for the dashboard

Interpret the event markers as candidate stress periods. If a hybrid model flattens its error line faster after a Fed or VIX shock than the linear VAR, that is evidence it is capturing nonlinear adjustment dynamics in PHP FX rather than merely fitting in-sample noise.